# Generative AI: Corporate Document Retrieval-Augmented Generation (RAG)

## Executive Summary
Large-scale retailers rely on strict adherence to operational policies, supply chain ethics, and customer service guidelines. When store managers or corporate staff have questions, they currently rely on manual keyword searches through massive PDF handbooks.

This notebook engineers a foundational **Retrieval-Augmented Generation (RAG)** architecture. We will use modern LangChain frameworks and open-source embedding models to transform a corporate handbook into a searchable vector database.

**Commercial Objective:** Build an AI-driven semantic search engine that understands the *meaning* behind an employee's question and instantly retrieves the exact operational policy required to answer it.

In [ ]:
# 1. Dependency Installation (Using '-q' for a quiet, clean installation)
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers

# 2. Setup and Imports
import warnings
warnings.filterwarnings('ignore') # Suppresses standard Python warnings for a clean output

# Modern LangChain Imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

print("✅ Modern LangChain & Vector Database Environment Initialised.")

## 1. Corporate Knowledge Base Simulation
To simulate a real-world scenario without relying on proprietary company data, we will generate a synthetic "Retail Operations & ESG Compliance" document.

This text covers distinct topics: Returns Policies, Ethical Sourcing (Sustainability), and Cybersecurity protocols for Point-of-Sale (POS) systems.

In [ ]:
# Simulated Corporate Handbook Content
corporate_handbook_text = """
SECTION 1: CUSTOMER RETURNS AND REFUNDS POLICY
All retail store managers must adhere to the 30-day return policy. Items must be returned in their original packaging with a valid receipt. If the item was purchased online, the customer may return it in-store, provided they present the digital order confirmation. For demi-fine jewellery, hygienic seals on earrings must remain unbroken for a full refund to be issued.

SECTION 2: ENVIRONMENTAL, SOCIAL, AND GOVERNANCE (ESG) - ETHICAL SOURCING
Our brand is committed to 100% traceability. All gold and silver used in our manufacturing process must be 100% recycled. Suppliers must provide annual audit certificates proving compliance with the Responsible Jewellery Council (RJC) standards. Diamonds must be conflict-free and adhere to the Kimberley Process. Any supplier failing a routine audit will be given a 60-day remediation period before contract termination.

SECTION 3: POINT OF SALE (POS) CYBERSECURITY PROTOCOLS
To maintain GDPR compliance and protect consumer financial data, all store staff must lock their POS terminals when stepping away from the till. USB drives are strictly prohibited from being inserted into any company hardware. In the event of a suspected network breach or malware detection, staff must immediately disconnect the terminal from the Wi-Fi and contact the IT Security Operations Centre (SOC) on extension 404.
"""

print("✅ Corporate Document Ingested.")

## 2. Semantic Chunking (Text Splitting)
Large Language Models (LLMs) and embedding models have "context limits" (they can only read a certain amount of text at a time).

We use LangChain's `RecursiveCharacterTextSplitter` to break our large handbook into smaller, digestible chunks. Crucially, we add a `chunk_overlap` so that a sentence isn't accidentally cut in half, preserving the human context.

In [ ]:
# Initialise the Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # Number of characters per chunk
    chunk_overlap=50,     # Overlap to maintain context between chunks
    length_function=len,
    separators=["\n\n", "\n", " ", ""] # Splits by paragraph first, then sentences
)

# Create LangChain Document objects
raw_document = [Document(page_content=corporate_handbook_text)]
chunked_documents = text_splitter.split_documents(raw_document)

print(f"✅ Document successfully split into {len(chunked_documents)} semantic chunks.")

# Display the first two chunks to verify the overlap logic
print("\n--- Chunk 1 Preview ---")
print(chunked_documents[0].page_content)
print("\n--- Chunk 2 Preview ---")
print(chunked_documents[1].page_content)

## 3. Vector Embeddings & FAISS Database Creation
This is where the mathematical magic happens. We use a HuggingFace embedding model (`all-MiniLM-L6-v2`) to read our text chunks and convert them into high-dimensional vectors.

We then load these vectors into a **FAISS (Facebook AI Similarity Search)** database. This acts as our AI's long-term memory.

In [ ]:
print("Downloading open-source embedding model (This may take a moment on first run)...")

# Initialise the HuggingFace Embedding Model
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build the FAISS Vector Database
vector_database = FAISS.from_documents(chunked_documents, embeddings_model)

print("✅ FAISS Vector Database successfully compiled and indexed.")

## 4. The Retrieval Engine (Semantic Search)
With our database built, we can now ask it questions in plain English.

Unlike a standard `CTRL+F` keyword search, Semantic Search understands *intent*. Even if we don't use the exact words from the document, the algorithm calculates the geometric "cosine distance" between our question and the text chunks to find the right answer.

In [ ]:
def ask_corporate_assistant(query, db, top_k=2):
    """
    Takes a natural language query, searches the FAISS database,
    and returns the most relevant corporate policy chunks.
    """
    print(f"\n👤 EMPLOYEE QUESTION: '{query}'")
    print("-" * 50)

    # Perform Similarity Search
    retrieved_chunks = db.similarity_search(query, k=top_k)

    print("🤖 AI RETRIEVED CONTEXT:")
    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f"\n[Source Document {i}]:")
        print(chunk.page_content)

    return retrieved_chunks

# --- Commercial Testing ---

# Test 1: Testing the ESG/Sustainability knowledge
ask_corporate_assistant("What are our rules regarding where we get our diamonds and gold?", vector_database)

# Test 2: Testing the Cybersecurity knowledge
ask_corporate_assistant("What should I do if I think the store till has a virus?", vector_database)

## 5. Limitations & Future Work
While this Semantic Retrieval pipeline successfully isolates the correct operational knowledge, a full production-grade RAG deployment would require the following enhancements:
* **LLM Integration (Generation Step):** Currently, the system returns the raw text chunks. In production, these retrieved chunks would be injected into the prompt of a Large Language Model (e.g., GPT-4 or Claude 3) with strict system instructions to generate a conversational, summarised answer *strictly* based on the provided context (to mitigate AI hallucination).
* **Multi-Modal Parsing:** Corporate knowledge is often locked in complex tables, charts, and scanned images. Integrating Optical Character Recognition (OCR) and specialized parsers (like Unstructured.io) would be necessary to ingest visual PDFs.
* **Vector Database Scalability:** For an enterprise with millions of documents, a local FAISS index would be migrated to a cloud-native, scalable vector database such as Pinecone, Weaviate, or pgvector.

## Conclusion
This Generative AI architecture successfully bridges the gap between unstructured corporate data and immediate operational intelligence.

By leveraging semantic embeddings rather than keyword matching, the system correctly understood that a "virus" relates to "malware" and "gold" relates to "ethical sourcing."

**Commercial Application:** Deploying this RAG architecture via a Slack or Microsoft Teams chatbot would allow retail staff to instantly resolve customer policy disputes, IT security protocols, and compliance questions without needing to escalate tickets to corporate HR or IT departments.